# 02 — DBpedia semantic triplets

**Purpose:** Builds a semantic-triplet dataset from successful DBpedia queries for training SPARQL embeddings.

**Input:** `../successful_queries/successful_queries_dbpedia.json` (produced by `01_dbpedia_query_collection.ipynb`).

**Output:** `../semantic_triplets/semantic_triplets_dbpedia.json` (anchor / positive / negative query triplets based on result-set Jaccard similarity).

**Run after:** `01_dbpedia_query_collection.ipynb`.

In [2]:
import json
import numpy as np
from SPARQLWrapper import SPARQLWrapper, JSON as SPARQL_JSON

endpoint = SPARQLWrapper("https://dbpedia.data.dice-research.org/sparql")
endpoint.setReturnFormat(SPARQL_JSON)

INPUT_JSON = "../successful_queries/successful_queries_dbpedia.json"
MAX_QUERIES = 50_000

def _norm(q: str) -> str:
    return "\n".join(line.strip() for line in q.strip().splitlines())

try:
    import ijson
    seen = set()
    unique_queries = []
    with open(INPUT_JSON, "rb") as f:
        for item in ijson.items(f, "item"):
            if len(unique_queries) >= MAX_QUERIES:
                break
            q = item.get("sparql_query") or item.get("query")
            if not q:
                continue
            q_norm = _norm(q)
            if q_norm not in seen:
                seen.add(q_norm)
                unique_queries.append(q)
    print(f"Streamed file: {len(unique_queries)} unique queries (stopped at {MAX_QUERIES})")
except ImportError:
    with open(INPUT_JSON, "r", encoding="utf-8") as f:
        raw_data = json.load(f)
    seen = set()
    unique_queries = []
    for item in raw_data:
        if len(unique_queries) >= MAX_QUERIES:
            break
        q = item.get("sparql_query") or item.get("query")
        if q and _norm(q) not in seen:
            seen.add(_norm(q))
            unique_queries.append(q)
    print(f"Loaded file: {len(unique_queries)} unique queries (install ijson for large files)")

queries = unique_queries[:MAX_QUERIES]
print(f"Using {len(queries)} queries for triplets (first 50k unique)")

answer_cache = {}

def normalize_query(q: str) -> str:
    return "\n".join(line.strip() for line in q.strip().splitlines())

def get_answers(query, var_name=None, limit=200):
    q_norm = normalize_query(query)
    if q_norm in answer_cache:
        return answer_cache[q_norm]

    endpoint.setQuery(query + f"\nLIMIT {limit}")
    try:
        res = endpoint.query().convert()
        bindings = res.get("results", {}).get("bindings", [])
    except Exception:
        bindings = []

    if not bindings:
        answers = set()
    else:
        vars_ = res.get("head", {}).get("vars", [])
        answers = set()
        for b in bindings:
            vals = []
            for v in vars_:
                if v in b:
                    vals.append(b[v]["value"])
            if vals:
                answers.add("|".join(vals))
    answer_cache[q_norm] = answers
    return answer_cache[q_norm]

SEMANTIC_POSITIVE_JACCARD_MIN = 0.8
SEMANTIC_NEGATIVE_JACCARD_MAX = 0.1
MAX_POSITIVES_PER_ANCHOR = 20
MAX_NEGATIVES_PER_ANCHOR = 20

semantic_items = []
n = len(queries)

for i in range(n):
    anchor_q = queries[i]
    A = get_answers(anchor_q)
    if not A:
        continue

    pos_list = []
    neg_list = []

    for j in range(n):
        if i == j:
            continue
        other_q = queries[j]
        B = get_answers(other_q)
        if not B:
            continue

        inter = len(A & B)
        union = len(A | B) or 1
        jacc = inter / union

        if jacc >= SEMANTIC_POSITIVE_JACCARD_MIN:
            pos_list.append({"query": other_q, "answer_jaccard": jacc})
        elif jacc <= SEMANTIC_NEGATIVE_JACCARD_MAX:
            neg_list.append({"query": other_q, "answer_jaccard": jacc})

    pos_list = sorted(pos_list, key=lambda x: -x["answer_jaccard"])[:MAX_POSITIVES_PER_ANCHOR]
    neg_list = sorted(neg_list, key=lambda x: x["answer_jaccard"])[:MAX_NEGATIVES_PER_ANCHOR]

    if pos_list and neg_list:
        semantic_items.append({
            "anchor_query": anchor_q,
            "positive_queries": pos_list,
            "negative_queries": neg_list,
        })

    if (i + 1) % 500 == 0:
                print(f"  Processed {i + 1}/{n} anchors, triplets so far: {len(semantic_items)}")

with open("../semantic_triplets/semantic_triplets_dbpedia.json", "w", encoding="utf-8") as f:
    json.dump(semantic_items, f, indent=2, ensure_ascii=False)

print(f"\nSaved {len(semantic_items)} semantic triplets to ../semantic_triplets/semantic_triplets_dbpedia.json")

sem = semantic_items
print("Entries:", len(sem))

if not sem:
    print("No semantic triplets produced. Check thresholds or query results.")
else:
    print("Example keys:", sem[0].keys())

    num_pos = sum(len(x["positive_queries"]) for x in sem)
    num_neg = sum(len(x["negative_queries"]) for x in sem)
    print("Total positives:", num_pos)
    print("Total negatives:", num_neg)
    print("Avg positives/anchor:", num_pos / len(sem))
    print("Avg negatives/anchor:", num_neg / len(sem))

    pos_j = [p["answer_jaccard"] for x in sem for p in x["positive_queries"]]
    neg_j = [n["answer_jaccard"] for x in sem for n in x["negative_queries"]]
    print("Pos Jaccard: min", np.min(pos_j), "max", np.max(pos_j), "mean", np.mean(pos_j))
    print("Neg Jaccard: min", np.min(neg_j), "max", np.max(neg_j), "mean", np.mean(neg_j))

    print("\nFirst 3 anchors (preview):")
    for i in range(min(3, len(sem))):
        x = sem[i]
        print("=" * 80)
        print("Anchor:")
        print(x["anchor_query"][:120].replace("\n", " "), "...")
        print("\nPositives:")
        for p in x["positive_queries"][:3]:
            print(f"  J={p['answer_jaccard']:.2f}", p["query"][:80].replace("\n", " "), "...")
        print("Negatives:")
        for n in x["negative_queries"][:3]:
            print(f"  J={n['answer_jaccard']:.2f}", n["query"][:80].replace("\n", " "), "...")


Streamed file: 50000 unique queries (stopped at 50000)
Using 50000 queries for triplets (first 50k unique)
Executing 50000 queries on endpoint...
  Cached 500/50000
  Cached 1000/50000
  Cached 1500/50000
  Cached 2000/50000
  Cached 2500/50000
  Cached 3000/50000
  Cached 3500/50000
  Cached 4000/50000
  Cached 4500/50000
  Cached 5000/50000
  Cached 5500/50000
  Cached 6000/50000
  Cached 6500/50000
  Cached 7000/50000
  Cached 7500/50000
  Cached 8000/50000
  Cached 8500/50000
  Cached 9000/50000
  Cached 9500/50000
  Cached 10000/50000
  Cached 10500/50000
  Cached 11000/50000
  Cached 11500/50000
  Cached 12000/50000
  Cached 12500/50000
  Cached 13000/50000
  Cached 13500/50000
  Cached 14000/50000
  Cached 14500/50000
  Cached 15000/50000
  Cached 15500/50000
  Cached 16000/50000
  Cached 16500/50000
  Cached 17000/50000
  Cached 17500/50000
  Cached 18000/50000
  Cached 18500/50000
  Cached 19000/50000
  Cached 19500/50000
  Cached 20000/50000
  Cached 20500/50000
  Cached 2100